# VAR -  Vector Autoregression Model
http://machinelearningplus.com/time-series/vector-autoregression-examples-python/ <br>
- <b> Autoregressive model </b> --> each variable (Time Series) is modeled as a function of the past values, that is the predictors are nothing but the lags (time delayed value) of the series.
- <b> Bidirectional model </b> --> predictors influence Y AND Y influence predictors = variables influence each other

## Import + setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.config import set_seeds, PLOTS_DIR
import src.pipeline as pipe
import src.models as mod
import src.evaluation as eval
import src.reporting as rep
import src.visualization as visual
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from itertools import permutations
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [3]:
set_seeds()

Random seeds set to 42. Deterministic operations enabled.


## Upload

In [4]:
df = pipe.load_data()
display(df.head())

Dropped 9 unusable indicators.
Dataset loaded: 65 years (from 1960 to 2024), 27 variables.


,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1960-01-01,40.639,NaN,20400656.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,40.144,-0.557139,20287312.0,NaN,NaN,NaN,NaN,NaN,70.324028,206830.0,...,72.707716,70.91,85.93,93.68,13933400.0,2181.5,NaN,NaN,NaN,NaN
1962-01-01,39.645,-0.574190,20171158.0,NaN,NaN,NaN,NaN,NaN,70.218626,206520.0,...,70.071359,72.59,86.90,94.43,14433210.0,2225.3,NaN,NaN,2.563660,16.566057
1963-01-01,39.147,-0.534554,20063620.0,NaN,NaN,NaN,NaN,NaN,69.735813,205100.0,...,63.883735,65.95,86.37,97.19,13324660.0,2115.2,NaN,NaN,2.651714,14.571604
1964-01-01,38.650,-0.455075,19972523.0,NaN,NaN,NaN,NaN,NaN,69.572609,204620.0,...,64.593354,69.57,90.28,101.31,14007520.0,2243.0,NaN,NaN,2.792923,15.115329


Each variable is modeled as a linear combination of past values of itself and the past values of other variables in the system. Since you have multiple time series that influence each other, it is modeled as a system of equations with one equation per variable (time series).

1. Analyze the time series characteristics
2. Test for causation amongst the time series
3. Test for stationarity
4. Transform the series to make it stationary, if needed
5. Find optimal order (p)
6. Prepare training and test datasets
7. Train the model
8. Roll back the transformations, if any.
9. Evaluate the model using test set
10. Forecast to future

- stationary
- seasonality
- structural breaks
- [spurious correlation](https://statisticsbyjim.com/basics/spurious-correlation/): a spurious correlation occurs when two variables are correlated but don’t have a causal relationship. In other words, it appears like values of one variable cause changes in the other variable, but that’s not actually happening. 

In [5]:
# def make_stationary(series):
#     def is_stationary(series):
#         result = adfuller(series.dropna())
#         return result[1] < 0.05  # p-value < 0.05 → stazionaria
    
#     if is_stationary(series) : 
#         #print("Serie stazionaria")
#         return series, 0
    
#     current_series = series.copy()
#     for d in range(1, 3):
#         current_series = current_series.diff()
#         if is_stationary(current_series):
#             return current_series, d
    
#     return current_series, 2

- test di stazionarietà = _ADF_ --> rendo tutte le time serie stazionarie + divido tra quelle stazionarie originariamente e quelle che hanno avuto di bisogno di 1 o 2 differenziazioni. 

**Questo ci porterà a 3 gruppi di time series:**
1. stazionarie <br>
2. 1-diff-transformed  -> non sono cointegrate = VAR standard sulle differenze <br>
-> sono cointegrate = VECM (Vector Error Correction Model) <br>
3. 2-diff-transformed <br>

- test di cointegrazione = _Johansen_ --> sulle 1-diff-transformed
    - prerequisiti: no multicollinearità e ritardi p appropriati
    - se r = 0 : nessuna cointegrazione --> VAR standard su tutte e tre i gruppi di prima
    - se r > 0 : sì cointegrazione --> VECM su variabili I(1) [si possono includere I(0), NON possono essere incluse I(2)]

## Loading of Integration Orders

In [6]:
config_path = "../results/integration_orders.xlsx"
if os.path.exists(config_path):
    print(f"Loading configuration from: {config_path}")
    config_df = pd.read_excel(config_path)
    if 'Indicator' in config_df.columns:
        config_df.set_index('Indicator', inplace=True)
    print("Configuration loaded correctly.")
else:
    print(f"ERROR: file {config_path} not found.")

Loading configuration from: ../results/integration_orders.xlsx
Configuration loaded correctly.


In [7]:
def prepare_data_for_var(df, config_df):
    original_I1_series = {}
    original_I2_series = {}
    df_stationary = pd.DataFrame(index=df.index)
    integration_map = {} 

    for col in df.columns:
        try:
            d = int(config_df.loc[col, 'd_full'])
        except IndexError:
            print(f"Warning: Ordine d non trovato per {col}, salto.")
            continue
        integration_map[col] = d
        series = df[col]
        
        if d == 0:
            df_stationary[col] = series
        elif d == 1:
            original_I1_series[col] = series 
            df_stationary[col] = series.diff()
        elif d == 2:
            original_I2_series[col] = series
            df_stationary[col] = series.diff().diff()
        else:
            pass
    df_stationary.dropna(inplace=True)
    return df_stationary, integration_map

## DataFrame containing only stationary series

In [8]:
df_stationary, transformation_info = prepare_data_for_var(df, config_df)
display(df_stationary.head())

,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1992-01-01,-0.036,-0.169583,-7607.0,-0.438475,-0.544846,-0.237835,0.265037,779.5,-0.258407,-760.0,...,23.883980,2.09,1.25,0.43,671737.0,341.1,-0.121136,7.949339e+08,-0.014675,-0.211437
1993-01-01,-0.060,-0.079171,-22547.0,-0.921662,-0.685508,-1.370476,0.265037,779.5,-0.231206,-680.0,...,69.294369,1.24,-2.98,-5.26,-119439.0,143.5,-0.119202,-9.238378e+09,-0.018946,0.215884
1994-01-01,-0.060,-0.041092,-30262.0,-0.184827,-0.099269,-0.347767,0.265037,779.5,-0.707218,-2080.0,...,-22.354164,0.33,-0.96,-1.73,-584956.0,-175.6,-0.006959,9.517599e+08,0.039501,0.681158
1995-01-01,-0.060,-0.019110,-33807.0,-0.314871,-0.255441,-0.424853,0.265037,779.5,-1.254633,-3690.0,...,-1.555090,3.83,-0.81,-3.17,492552.0,-7.9,-0.013877,2.096193e+09,0.006140,0.038495
1996-01-01,-0.060,0.026185,-28831.0,-0.454062,-0.263117,-0.800961,0.265037,779.5,0.054401,160.0,...,46.442966,1.19,2.99,3.95,1220922.0,280.9,-0.001888,4.061960e+09,-0.031885,-0.869866


In [9]:
corr_matrix = df_stationary.dropna().corr().abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

CORRELATED_COUPLES = [
    column for column in upper_triangle.columns 
    if any(upper_triangle[column] > 0.99)
]

if CORRELATED_COUPLES:
    print("ATTENZIONE: high multicollinearity found (> 0.99)")
    print("Problematic couples:")

    for col in upper_triangle.columns:
        highly_corr = upper_triangle.index[upper_triangle[col] > 0.99].tolist()
        if highly_corr:
            print(f"- {col} = {highly_corr}")
else:
    print("No perfect multicollinearity (>0.99) found.")

ATTENZIONE: high multicollinearity found (> 0.99)
Problematic couples:
- agriland_abs = ['agriland_percent']
- arableland_person = ['arableland_percent']
- arableland_abs = ['arableland_percent', 'arableland_person']


In [10]:
REDUNDANT = [
    'forestarea_abs',
    'agriland_abs', 
    'arableland_abs', 
    'arableland_person',
    'fertilizer_abs',
    'valueadded_dollars',
    'population_abs',
    'employment_male',
    'employment_female'
]

CATEGORIZED_VARS = {
    'demography': [
        'population_percent',
        'population_growth',
        'employment_tot'
    ],
    'land_use': [
        'forestarea_percent',
        'agriland_percent',
        'arableland_percent',
        'cerealland_abs',
        'cropland_percent'
    ],
    'prerequisites': [
        'withdrawals_percent',
        'fertilizer_percent'
    ],
    'production': [
        'livestock_production_index',
        'food_production_index',
        'crop_production_index',
        'cereal_production',
        'cerealyield_abs'
    ],
    'economy': [
        'valueadded_percent',
        'exports_percent',
        'imports_percent'
    ],
    'intersectoral1' : [
        'population_percent',
        'agriland_percent',
        'fertilizer_percent',
        'food_production_index',
        'valueadded_percent'
    ],
    'intersectoral2' : [ 
        'employment_tot', 
        'imports_percent',
        'cerealland_abs',
        'fertilizer_percent',
        'withdrawals_percent'
    ]
}

In [11]:
df_I1_johansen = df[[col for col in df.columns if transformation_info.get(col) == 1]]
cointegration_results = {}

for category_name, var_list in CATEGORIZED_VARS.items():
    print(f"\n=======================================================")
    print(f"CATEGORY: {category_name.upper()}")
    print(f"=======================================================")
    
    # Seleziona le vars I(1) pulite per questo gruppo
    # (intersezione tra le vars I(1) e quelle della categoria)
    vars_to_test = [col for col in var_list if col in df_I1_johansen.columns]
    
    if len(vars_to_test) < 2:
        print(f"Test saltato: meno di 2 vars I(1) in questo gruppo ({vars_to_test}).")
        cointegration_results[category_name] = {'rank': 0, 'status': 'Saltato - Poche vars I(1)', 'vars': vars_to_test}
        continue
        
    print(f"vars I(1) in test: {vars_to_test}")
    subgroup_df = df_I1_johansen[vars_to_test].dropna()
    
    if subgroup_df.shape[0] < 20:
        print(f"Test saltato: dati insufficienti dopo dropna ({subgroup_df.shape[0]} righe).")
        cointegration_results[category_name] = {'rank': 0, 'status': 'Saltato - Dati insuff.', 'vars': vars_to_test}
        continue

    try:
        var_model_sub = VAR(subgroup_df)
        
        safe_maxlags = min(2, subgroup_df.shape[0] - 1)
        safe_maxlags = max(0, safe_maxlags)
        
        selected_result = var_model_sub.select_order(maxlags=safe_maxlags)
        p_lags = selected_result.selected_orders['aic']
        if p_lags == 0:
            p_lags = 1
        k_ar_diff = p_lags - 1
        print(f"Ritardo ottimale (p): {p_lags} | k_ar_diff (p-1): {k_ar_diff}")
    except Exception as e:
        print(f"Errore selezione ritardi, uso k_ar_diff=1 (p=2). Errore: {e}")
        k_ar_diff = 1 # Fallback
        
    try:
        johansen_result = coint_johansen(
            subgroup_df,
            det_order=1,
            k_ar_diff=k_ar_diff
        )

        trace_rank = 0
        significance_level = 1 # 5%
        
        print("H0: r <= k  | Stat. Traccia | Val. Critico (5%) | Decisione")
        
        for i in range(len(subgroup_df.columns)):
            stat = johansen_result.lr1[i]
            crit = johansen_result.cvt[i, significance_level]

            if np.isnan(crit):
                print(f"H0: r <= {i:<2} | {stat:<13.2f} | {crit:<17.2f} | ERRORE (nan)")
                break
            
            decision = "Rifiuta H0"
            if stat > crit:
                trace_rank = i + 1
            else:
                decision = "Non Rifiuta H0"
            
            print(f"H0: r <= {i:<2} | {stat:<13.2f} | {crit:<17.2f} | {decision}")
            
            if decision == "Non Rifiuta H0":
                break
                
        final_rank = trace_rank
        print(f"\n--> rank di Cointegrazione (r) per '{category_name}': {final_rank}")
        
        if final_rank > 0:
            print("--> CONCLUSIONE: Le serie in questo gruppo sono COINTEGRATE (usare VECM).")
            cointegration_results[category_name] = {'rank': final_rank, 'status': 'Cointegrato', 'vars': vars_to_test}
        else:
            print("--> CONCLUSIONE: Le serie non sono cointegrate (usare VAR in differenze).")
            cointegration_results[category_name] = {'rank': 0, 'status': 'Non Cointegrato', 'vars': vars_to_test}

    except np.linalg.LinAlgError:
        print("ERRORE: LinAlgError (Matrice non definita positiva).")
        print("--> Causa probabile: multicollinearità residua o dati insufficienti.")
        print("--> CONCLUSIONE: Tratto come Non Cointegrato (r=0).")
        cointegration_results[category_name] = {'rank': 0, 'status': 'Errore (LinAlg)', 'vars': vars_to_test}

print("\n=======================================================")
print("RIEPILOGO TEST DI COINTEGRAZIONE")
print("=======================================================")
print(f"{'Categoria':<15} | {'rank (r)':<8} | {'Status':<20} | vars Testate")
print("-" * 80)

for cat, res in cointegration_results.items():
    vars_str = ', '.join(res['vars']) if res['vars'] else 'N/A'
    print(f"{cat:<15} | {res['rank']:<8} | {res['status']:<20} | {vars_str}")


CATEGORY: DEMOGRAPHY
vars I(1) in test: ['population_percent', 'population_growth', 'employment_tot']
Ritardo ottimale (p): 2 | k_ar_diff (p-1): 1
H0: r <= k  | Stat. Traccia | Val. Critico (5%) | Decisione
H0: r <= 0  | 40.73         | 35.01             | Rifiuta H0
H0: r <= 1  | 8.85          | 18.40             | Non Rifiuta H0

--> rank di Cointegrazione (r) per 'demography': 1
--> CONCLUSIONE: Le serie in questo gruppo sono COINTEGRATE (usare VECM).

CATEGORY: LAND_USE
vars I(1) in test: ['forestarea_percent', 'agriland_percent', 'arableland_percent', 'cerealland_abs', 'cropland_percent']
Ritardo ottimale (p): 1 | k_ar_diff (p-1): 0
H0: r <= k  | Stat. Traccia | Val. Critico (5%) | Decisione
H0: r <= 0  | 53.59         | 79.34             | Non Rifiuta H0

--> rank di Cointegrazione (r) per 'land_use': 0
--> CONCLUSIONE: Le serie non sono cointegrate (usare VAR in differenze).

CATEGORY: PREREQUISITES
vars I(1) in test: ['withdrawals_percent', 'fertilizer_percent']
Ritardo ottima

- r = 0 --> non hanno una relazione di equilibrio stabile nel lungo periodo. Anche se vagano (essendo I(1)), non sono "legate" l'una all'altra. Posso usare VAR.
- r > 0 --> hanno una relazione di equilibrio stabile nel lungo periodo. 
    - Questo gruppo di 5 variabili è "legato" da 3 relazioni di lungo periodo (guinzagli). Anche se le singole serie vagano, queste 3 relazioni le costringono a muoversi insieme nel tempo. Queste variabili devono essere modellate usando un VECM (Vector Error Correction Model), che modella sia le dinamiche di breve periodo (le differenze) sia il ritorno all'equilibrio di lungo periodo (l'Error Correction Term).

In [12]:
# teniamo vars non cointegrate che devono essere studiate da VAR
suitable_for_var = []
for cat, res in cointegration_results.items():
    if res['rank'] == 0:
        suitable_for_var.extend([v for v in res['vars'] if v not in suitable_for_var])

# prendo tutte le serie rese stazionarie
# tengo solo quelle non cointegrate
df_var_input = df_stationary.copy()
df_var_input = df_var_input[suitable_for_var]
df_var_input = df_var_input.dropna()

print(df_var_input.shape)
display(df_var_input.head())

(30, 9)


,forestarea_percent,agriland_percent,arableland_percent,cerealland_abs,cropland_percent,population_percent,fertilizer_percent,food_production_index,valueadded_percent
1992-01-01,0.265037,-0.258407,-0.510013,-176287.0,-0.241406,-0.036,23.883980,1.25,-0.121136
1993-01-01,0.265037,-0.231206,-0.680018,-149640.0,-0.166604,-0.060,69.294369,-2.98,-0.119202
1994-01-01,0.265037,-0.707218,-0.751420,27970.0,-0.054401,-0.060,-22.354164,-0.96,-0.006959
1995-01-01,0.265037,-1.254633,-0.156404,112500.0,-0.574615,-0.060,-1.555090,-0.81,-0.013877
1996-01-01,0.265037,0.054401,0.166604,7398.0,0.098603,-0.060,46.442966,2.99,-0.001888


In [13]:
max_lag_granger = 3

def test_granger_pair(data, causing_var, caused_var, maxlag):
    """
    Esegue il test di Granger per una singola coppia e restituisce il p-value minimo.
    H0: 'causing_var' NON causa (Granger) 'caused_var'
    """
    test_data = data[[caused_var, causing_var]]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            results = grangercausalitytests(test_data, maxlag=maxlag, verbose=False)
        except Exception as e:
            print(f"Errore testando {causing_var} -> {caused_var}: {e}")
            return np.nan

    # Estraiamo il p-value minimo tra tutti i ritardi testati
    # Ci interessa il test 'ssr_ftest' (colonna 1)
    min_p_value = 1.0
    for lag in range(1, maxlag + 1):
        p_value = results[lag][0]['ssr_ftest'][1]
        if p_value < min_p_value:
            min_p_value = p_value
    return min_p_value

variables = df_var_input.columns
granger_results = []

# 'permutations' testa sia (A, B) che (B, A)
for pair in permutations(variables, 2):
    causing_var = pair[0]
    caused_var = pair[1]

    p_value = test_granger_pair(df_var_input, causing_var, caused_var, max_lag_granger)
    
    granger_results.append({
        "Variabile Causa (X)": causing_var,
        "Variabile Effetto (Y)": caused_var,
        "Min P-Value": p_value
    })

df_granger_summary = pd.DataFrame(granger_results)
df_granger_summary = df_granger_summary.sort_values(by="Min P-Value")
display(df_granger_summary.head(10))

,Variabile Causa (X),Variabile Effetto (Y),Min P-Value
63,food_production_index,valueadded_percent,0.000023
51,fertilizer_percent,cerealland_abs,0.000137
62,food_production_index,fertilizer_percent,0.004824
10,agriland_percent,cerealland_abs,0.013565
70,valueadded_percent,fertilizer_percent,0.014656
28,cerealland_abs,population_percent,0.018295
29,cerealland_abs,fertilizer_percent,0.019235
18,arableland_percent,cerealland_abs,0.026807
40,population_percent,forestarea_percent,0.027053
11,agriland_percent,cropland_percent,0.036278


*Domande che mi devo fare per coppie:*
1. Do they make sense as causal relationships? <br> 
2. Do they fit established theory? <br>
3. Can you find a mechanism for causation? <br>
4. Is there a direct link, or are mediator variables involved? <br>

*Scrematura logica dei risultati della Granger causality:*
- **Employment in agriculture - Agricultural raw materials imports** = meno persone lavorano nell'agricoltura più bisogna importare?
- **Employment in agriculture - Land under cereal production** = meno persone lavorano nell'agricoltura più diminuisce la superficie dei campi di cereali
- **Land under cereal production - Fertilizer consumption** = più aumenta la terra arata a cereali più aumenta il consumo di fertilizzanti (questo ci sta!!)
- Arable land - Land under cereal production = meno terra arabile, meno terra arata a cereali (boh sembra stupido)

In [14]:
def optimize_VARMAX(endog: pd.DataFrame, max_p_to_try: int) -> int:
    results = []
    for i in tqdm(range(1, max_p_to_try + 1), desc="Ottimizzazione AIC"):
        try:
            model = VARMAX(endog, order=(i, 0)).fit(disp=False)
            results.append({'p': i, 'aic': model.aic})
        except Exception:
            break
            
    if not results:
        return 1 # Fallback

    result_df = pd.DataFrame(results)
    best_p = result_df.loc[result_df['aic'].idxmin()]['p']
    return int(best_p)

Per ogni coppia che ho individuato:
1. trovo la p ottimale
2. stimo modello VARMAX(p,0)
3. faccio residual analysis
    - QQ plot: i residui sono "normali"? --> i punti devono seguire la linea rossa
    - ACF plot: i residui sono "non correlati"? --> le barre devono stare nell'area blu
    - Ljung-box test: versione numerica dell'ACF plot --> p-value deve essere > 0.05

In [15]:
CAUSEEFFECT_COUPLES = [
    ['food_production_index', 'valueadded_percent'],
    ['fertilizer_percent', 'cerealland_abs'],
    ['food_production_index', 'fertilizer_percent'],
    ['agriland_percent', 'cerealland_abs'],
    ['valueadded_percent', 'fertilizer_percent'],
    ['cerealland_abs','population_percent'],
    ['cerealland_abs', 'fertilizer_percent'],
    ['population_percent', 'forestarea_percent'],
    ['arableland_percent', 'cerealland_abs'],
    ['agriland_percent', 'cropland_percent']
]

_selected = []
for pair in CAUSEEFFECT_COUPLES:
    for v in pair:
        if v not in _selected:
            _selected.append(v)

# Filtra solo quelle presenti in df_var_input
found = [v for v in _selected if v in df_var_input.columns]
df_causeeffect = df_var_input[found].copy()

In [16]:
original_index = df_causeeffect.index
original_columns = df_causeeffect.columns

scaler = StandardScaler()
df_causeeffect_scaled_array = scaler.fit_transform(df_causeeffect)
df_causeeffect_scaled = pd.DataFrame(df_causeeffect_scaled_array, 
                                index=original_index, 
                                columns=original_columns)

display(df_causeeffect_scaled.head())

,food_production_index,valueadded_percent,fertilizer_percent,cerealland_abs,agriland_percent,population_percent,forestarea_percent,arableland_percent,cropland_percent
1992-01-01,0.426483,-0.808304,0.171645,-0.925689,0.168556,1.193439,1.303857,-0.436521,-0.549376
1993-01-01,-0.620947,-0.787301,0.476209,-0.734239,0.196680,0.952070,1.303857,-0.674302,-0.278148
1994-01-01,-0.120756,0.432107,-0.138471,0.541827,-0.295498,0.952070,1.303857,-0.774170,0.128695
1995-01-01,-0.083613,0.356951,0.001027,1.149146,-0.861503,0.952070,1.303857,0.058062,-1.757575
1996-01-01,0.857340,0.487197,0.322946,0.394024,0.491988,0.952070,1.303857,0.509845,0.683480


In [ ]:
TRAIN_LEN = int(len(df_causeeffect_scaled) * 0.8)
HORIZON = len(df_causeeffect_scaled) - TRAIN_LEN
WINDOW = 1

_pairs = []
for item in CAUSEEFFECT_COUPLES:
    _pairs.append((item[0], item[1]))

summary_results = {}
for causing, caused in _pairs:
    print(f"ANALISI COPPIA: '{causing}' -> '{caused}'")
    df_pair = df_causeeffect_scaled[[causing, caused]].dropna()[:TRAIN_LEN]
    n_obs = df_pair.shape[0]
    safe_max_p = max(1, TRAIN_LEN // 3)
    if safe_maxlags < 1:
        print(f"Pochi dati ({n_obs} osservazioni).")
        continue
    
    p_selected = optimize_VARMAX(df_pair, safe_maxlags)
    print(f"Ritardo ottimale (AIC) scelto (p): {p_selected}")
    fitted_model = VARMAX(df_pair, order=(p_selected, 0)).fit(disp=False)
    # print(fitted_model.summary())
    
    resid = fitted_model.resid
    
    ljung_pvals = {}
    for col in resid.columns:
        lb_df = acorr_ljungbox(resid[col].dropna(), lags=[10], return_df=True)
        pval = lb_df['lb_pvalue'].iloc[0]
        ljung_pvals[col] = pval
        print(f"Ljung-Box p-value ({col}): {pval:.4f}")
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"Analisi Residui per VAR({p_selected})", fontsize=20, fontweight='bold')

    qqplot(resid[causing].dropna(), line='s', ax=axes[0, 0])
    axes[0, 0].set_title(f"QQ-Plot Residui: {causing}")
    qqplot(resid[caused].dropna(), line='s', ax=axes[0, 1])
    axes[0, 1].set_title(f"QQ-Plot Residui: {caused}")

    plot_acf(resid[causing].dropna(), ax=axes[1, 0], lags=20)
    axes[1, 0].set_title(f"ACF Residui: {causing}")
    plot_acf(resid[caused].dropna(), ax=axes[1, 1], lags=20)
    axes[1, 1].set_title(f"ACF Residui: {caused}")
    
    plt.tight_layout()
    plt.show()
    
    summary_results[(causing, caused)] = {
        "p_selected": p_selected,
        "n_obs": n_obs,
        "ljungbox_pvals": ljung_pvals,
        "fitted_model": fitted_model
    }

ANALISI COPPIA: 'food_production_index' -> 'valueadded_percent'


Ottimizzazione AIC: 100%|██████████| 2/2 [00:01<00:00,  1.37it/s]


Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (food_production_index): 0.7908
Ljung-Box p-value (valueadded_percent): 0.9970


ANALISI COPPIA: 'fertilizer_percent' -> 'cerealland_abs'


Ottimizzazione AIC: 100%|██████████| 2/2 [00:01<00:00,  1.49it/s]


Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (fertilizer_percent): 0.7391
Ljung-Box p-value (cerealland_abs): 0.8870


ANALISI COPPIA: 'food_production_index' -> 'fertilizer_percent'


...

Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (population_percent): 0.9915
Ljung-Box p-value (forestarea_percent): 0.9846


ANALISI COPPIA: 'arableland_percent' -> 'cerealland_abs'


Ottimizzazione AIC: 100%|██████████| 2/2 [00:01<00:00,  1.87it/s]


Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (arableland_percent): 0.3723
Ljung-Box p-value (cerealland_abs): 0.5489


ANALISI COPPIA: 'agriland_percent' -> 'cropland_percent'


Ottimizzazione AIC: 100%|██████████| 2/2 [00:01<00:00,  1.81it/s]


Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (agriland_percent): 0.5077
Ljung-Box p-value (cropland_percent): 0.6029




In [25]:
def rolling_forecast(df, causing, caused, p, train_len, horizon, window, method):
    total_len = train_len + horizon
    
    if method == 'VAR':
        causing_pred_VAR = []
        caused_pred_VAR = []
        for i in tqdm(range(train_len, total_len, window)):
            try:
                scaled_test = df[[causing, caused]][:i]
                model = VARMAX(scaled_test, order=(p, 0))
                res = model.fit(disp=False)

                predictions_scaled = res.forecast(steps=window)
                preds_arr = np.asarray(predictions_scaled)

                if preds_arr.ndim == 1:
                    preds_arr = preds_arr.reshape(1, -1)

                oos_pred_causing = preds_arr[:, 0]
                oos_pred_caused = preds_arr[:, 1]

                causing_pred_VAR.extend(oos_pred_causing.tolist())
                caused_pred_VAR.extend(oos_pred_caused.tolist())

            except (np.linalg.LinAlgError, ValueError) as e:
                print(f"Errore stima a i={i}: {e}. Inserimento NaN.")
                causing_pred_VAR.extend([np.nan] * window)
                caused_pred_VAR.extend([np.nan] * window)
        return causing_pred_VAR, caused_pred_VAR

In [26]:
def inverse_scale_dataframe(df_pair_diff, scaler, original_scaler_columns, pair_key):
    df_pair_unscaled_diff = pd.DataFrame(index=df_pair_diff.index)
    causing, caused = pair_key
    
    # Mappatura indici colonne
    try:
        col_indices = {col: i for i, col in enumerate(original_scaler_columns)}
        idx_causing = col_indices[causing]
        idx_caused = col_indices[caused]
    except KeyError as e:
        print(f"Errore: variabile mancante nello scaler: {e}")
        return None

    n_samples = len(df_pair_diff)
    n_features = len(original_scaler_columns)
    
    # Matrice dummy riutilizzabile
    dummy_data = np.zeros((n_samples, n_features))

    for col_suffix in ["_actual", "_pred_VAR"]:
        col_causing = f"{causing}{col_suffix}"
        col_caused = f"{caused}{col_suffix}"
        
        # Popoliamo solo le colonne di interesse
        dummy_data[:, idx_causing] = df_pair_diff[col_causing].values
        dummy_data[:, idx_caused] = df_pair_diff[col_caused].values
        
        # Inversione
        inverted_matrix = scaler.inverse_transform(dummy_data)
        
        # Estrazione
        df_pair_unscaled_diff[col_causing] = inverted_matrix[:, idx_causing]
        df_pair_unscaled_diff[col_caused] = inverted_matrix[:, idx_caused]
        
        # Reset dummy per il prossimo ciclo (importante se le colonne si sovrappongono, qui no ma per sicurezza)
        dummy_data[:, idx_causing] = 0
        dummy_data[:, idx_caused] = 0
        
    return df_pair_unscaled_diff

In [29]:
def inverse_difference_dataframe(df_pair_unscaled_diff, wide_df, df_metadata, pair_key, train_len):
    """
    Ricostruisce i livelli originali dalle differenze, ancorandosi all'ultimo valore reale noto.
    """
    df_pair_level = df_pair_unscaled_diff.copy()
    causing, caused = pair_key
    
    first_pred_date = df_pair_unscaled_diff.index[0]
    
    try:
        loc_first_pred = wide_df.index.get_loc(first_pred_date)
        last_train_date = wide_df.index[loc_first_pred - 1]
    except KeyError:
        last_train_date = wide_df.index[wide_df.index < first_pred_date][-1]

    for var_name in [causing, caused]:
        d = df_metadata.get(var_name, 0) # Usa .get() per sicurezza
        last_known_level = wide_df.loc[last_train_date, var_name]
        
        last_known_diff1 = 0
        if d == 2:
            series_diff = wide_df[var_name].diff()
            last_known_diff1 = series_diff.loc[last_train_date]

        for col_suffix in ["_actual", "_pred_VAR"]:
            col_name = f"{var_name}{col_suffix}"
            
            preds_unscaled_diff = df_pair_unscaled_diff[col_name].values
            
            if d == 0:
                preds_level = preds_unscaled_diff
                
            elif d == 1:
                preds_level = last_known_level + np.cumsum(preds_unscaled_diff)
                
            elif d == 2:
                preds_diff1_level = last_known_diff1 + np.cumsum(preds_unscaled_diff)
                preds_level = last_known_level + np.cumsum(preds_diff1_level)
            
            df_pair_level[col_name] = preds_level
            
    return df_pair_level

In [47]:
original_scaler_columns = list(df_causeeffect.columns)
TRAIN_LEN = int(len(df_causeeffect_scaled) * 0.8)
HORIZON = len(df_causeeffect_scaled) - TRAIN_LEN
WINDOW = 1
BASELINE_METHOD = 'drift'

forecast_results_level = {}
scaled_diff_test = df_causeeffect_scaled.iloc[TRAIN_LEN : TRAIN_LEN + HORIZON]

for pair_key, results in summary_results.items():
    causing, caused = pair_key
    print(f"Processing pair: {causing} -> {caused}")
    
    optimal_p = results['p_selected']
    causing_pred_VAR, caused_pred_VAR = rolling_forecast(
        df_causeeffect_scaled,
        causing,
        caused,
        optimal_p,
        TRAIN_LEN, HORIZON, WINDOW, 'VAR'
    )       
    
    df_test_pair_scaled_diff = pd.DataFrame({
        f"{causing}_actual": scaled_diff_test[causing],
        f"{caused}_actual": scaled_diff_test[caused],
        f"{causing}_pred_VAR": causing_pred_VAR,
        f"{caused}_pred_VAR": caused_pred_VAR,
        f"{causing}_pred_baseline": np.zeros(len(scaled_diff_test)),
        f"{caused}_pred_baseline": np.zeros(len(scaled_diff_test)),
    }, index=scaled_diff_test.index)
    
    df_pair_unscaled = inverse_scale_dataframe(
        df_test_pair_scaled_diff, scaler, original_scaler_columns, pair_key
    )
    df_pair_level = inverse_difference_dataframe(
        df_pair_unscaled, df, transformation_info, pair_key, TRAIN_LEN
    )
    first_test_date = df_pair_level.index[0]
    for var_name in [causing, caused]:
        train_serie = df[var_name][df.index < first_test_date].dropna()
        baseline_series = mod.get_baseline_prediction(
            BASELINE_METHOD,
            train_serie,
            df_pair_level.index 
        )
        col_base_name = f"{var_name}_pred_baseline" 
        df_pair_level[col_base_name] = baseline_series.values
    forecast_results_level[pair_key] = df_pair_level

Processing pair: food_production_index -> valueadded_percent


100%|██████████| 6/6 [00:02<00:00,  2.03it/s]


Processing pair: fertilizer_percent -> cerealland_abs


100%|██████████| 6/6 [00:02<00:00,  2.41it/s]


Processing pair: food_production_index -> fertilizer_percent


...

100%|██████████| 6/6 [00:08<00:00,  1.34s/it]


Processing pair: arableland_percent -> cerealland_abs


100%|██████████| 6/6 [00:02<00:00,  2.77it/s]


Processing pair: agriland_percent -> cropland_percent


100%|██████████| 6/6 [00:02<00:00,  2.70it/s]


In [63]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import pandas as pd
import numpy as np
import textwrap
from src.config import PLOTS_DIR, COLORS, LINE_STYLES, REVERSE_VAR_NAMES

def plot_var_forecast_stacked(original_df, test_data_pair, 
                            causing_var, caused_var,
                            folder_name="06_VAR"):
    """
    Plotta i risultati del VAR.
    
    Args:
        original_df (pd.DataFrame): Il dataframe COMPLETO (originale) contenente tutta la storia.
        test_data_pair (pd.DataFrame): Il dataframe con i risultati del test (actual, pred, baseline).
        causing_var (str): Nome variabile causa.
        caused_var (str): Nome variabile effetto.
        folder_name (str): Cartella di destinazione.
    """
    save_folder = os.path.join(PLOTS_DIR, folder_name)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
        
    first_test_idx = test_data_pair.index[0]
    train_data = original_df[original_df.index < first_test_idx]
    last_train_idx = train_data.index[-1]
    
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 12), sharex=True)
    fig.suptitle(f"VAR Forecast: {causing_var} -> {caused_var}", fontsize=16, fontweight="bold")
    
    plot_configs = [
        (causing_var, axes[0]),
        (caused_var, axes[1])
    ]

    for var_name, ax in plot_configs:
        train_series = train_data[var_name]
        
        col_actual = f"{var_name}_actual"
        col_pred = f"{var_name}_pred_VAR"
        col_base = f"{var_name}_pred_baseline"
        
        test_actual = test_data_pair[col_actual]
        test_pred = test_data_pair[col_pred]
        test_baseline = test_data_pair[col_base]
        
        ax.plot(train_series.index, train_series.values, 
                label='Train', color=COLORS.get('train', 'grey'), marker='.', linewidth=2)
        
        ax.plot(test_actual.index, test_actual.values, 
                label='Test (Real)', color=COLORS.get('test_real', 'black'), marker='.', linewidth=2)
        
        ax.plot([last_train_idx, first_test_idx], [train_series.iloc[-1], test_actual.iloc[0]], 
                color=COLORS.get('train', 'grey'), linewidth=1.5)

        ax.plot(test_pred.index, test_pred.values, 
                linestyle=LINE_STYLES.get('pred', '--'), label='Pred (VAR)', 
                color=COLORS.get('pred', 'red'), linewidth=2)

        ax.plot(test_baseline.index, test_baseline.values, 
                linestyle=LINE_STYLES.get('baseline', ':'), label='Baseline (Last Val)', 
                color=COLORS.get('baseline', 'blue'))
        
        ax.axvspan(test_actual.index[0], test_actual.index[-1], color='#C3C3C3', alpha=0.3)
        
        long_name = REVERSE_VAR_NAMES.get(var_name, var_name)
        ax.set_title(f"Variable: {long_name}", fontsize=14, fontweight='bold')
        
        ax.set_ylabel("Value")
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend(loc='best')

    axes[1].set_xlabel("Year", fontsize=12)
    
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=0)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    filename = f"VAR_{causing_var}_{caused_var}.png"
    full_path = os.path.join(save_folder, filename)
    try:
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved: {full_path}")
    except Exception as e:
        print(f"Error saving plot: {e}")
    
    plt.close()

In [66]:
print("VALIDATION PHASE")

for (causing, caused), df_results in forecast_results_level.items():
    print(f"\nProcessing Pair: {causing} -> {caused}")
    plot_var_forecast_stacked(
        original_df=df, 
        test_data_pair=df_results,
        causing_var=causing,
        caused_var=caused,
        folder_name="06_VAR"
    )
    
    optimal_p = '?'
    if 'summary_results' in locals() and (causing, caused) in summary_results:
        optimal_p = summary_results[(causing, caused)]['p_selected']
    for var_name in [causing, caused]:
        other_var = caused if var_name == causing else causing
        col_actual = f"{var_name}_actual"
        col_pred = f"{var_name}_pred_VAR"
        
        y_true = df_results[col_actual].dropna()
        y_pred = df_results[col_pred].dropna()
        
        common_index = y_true.index.intersection(y_pred.index)
        y_true = y_true.loc[common_index]
        y_pred = y_pred.loc[common_index]

        first_test_date = y_true.index[0]
        y_train = df[var_name][df.index < first_test_date].dropna()

        pred_metrics = eval.compute_errors(y_true, y_pred)
        print(f"  -> {var_name}: RMSE={pred_metrics['RMSE']:.4f}, MAPE={pred_metrics['MAPE']:.4f}, R2={pred_metrics['R2']:.4f}")

        try:
            pred_residuals = eval.compute_residual_diagnostics(y_true, y_pred)
            print(f"     Residui:  Shapiro P={pred_residuals.get('shapiro_wilk_pvalue', 0):.4f}",
                f"Ljung-Box P={pred_residuals.get('ljung_box_pvalue', 0):.4f}")
        except Exception as e:
            print(f"     Residual diagnostics failed: {e}")
            pred_residuals = {}

        try:
            visual.plot_residuals(
                y_true=y_true,
                y_pred=y_pred,
                variable_name=var_name,
                model_name=f"VAR{var_name}_with_{other_var}", # Nome descrittivo
                folder_name="06_residVAR"
            )
        except Exception as e:
            print(f"     Residual plot failed: {e}")

        rep.save_experiment_results(
            indicator=var_name,
            model_name='VAR',
            configuration=f"with{other_var}_p={optimal_p}", # Config specifica
            y_test=y_true.values,
            y_pred=y_pred.values,
            years_test=y_true.index.year.tolist(),
            y_train=y_train.values,
            params={'p': optimal_p, 'partner': other_var},
            training_time=0 
        )

print("\nVAR Validation & Logging Completed.")

VALIDATION PHASE

Processing Pair: food_production_index -> valueadded_percent
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_VAR\VAR_food_production_index_valueadded_percent.png
  -> food_production_index: RMSE=1.3014, MAPE=1.2100, R2=-0.0804
     Residui:  Shapiro P=0.4617 Ljung-Box P=0.8670
Diagnostics saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_residVAR\RESIDVARfood_production_index_with_valueadded_percent_food_production_index.png
Saving results for VAR | food_production_index...
Leaderboard updated: food_production_index | VAR
Save leaderboard complete.
  -> valueadded_percent: RMSE=0.1485, MAPE=6.5900, R2=-15.1040
     Residui:  Shapiro P=0.9195 Ljung-Box P=0.5488
Diagnostics saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_residVAR\RESIDVARvalueadded_percent_with_food_production_index_valueadded_percent.png
Saving results for VAR | valueadded_percent...
Leaderboard 

## Future Forecasts 2030

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import pandas as pd
import numpy as np
from src.config import PLOTS_DIR, COLORS, LINE_STYLES, REVERSE_VAR_NAMES

def plot_var_future_stacked(history_df, future_df, 
                            causing_var, caused_var,
                            folder_name="06_FutureVAR"):
    """
    Plotta le previsioni future (fino al 2030) per una coppia VAR.
    Stile identico a plot_var_forecast_stacked ma senza la linea 'Test Actual'.
    """
    save_folder = os.path.join(PLOTS_DIR, folder_name)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
        
    last_hist_idx = history_df.index[-1]
    first_future_idx = future_df.index[0]
    
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 12), sharex=True)
    fig.suptitle(f"VAR Future Forecast 2030: {causing_var} -> {caused_var}", fontsize=16, fontweight="bold")
    
    plot_configs = [
        (causing_var, axes[0]),
        (caused_var, axes[1])
    ]

    for var_name, ax in plot_configs:
        # Dati Storici
        hist_series = history_df[var_name]
        
        # Dati Futuri
        col_pred = f"{var_name}_pred_VAR"
        col_base = f"{var_name}_pred_baseline"
        
        future_pred = future_df[col_pred]
        future_base = future_df[col_base]
        
        ax.plot(hist_series.index, hist_series.values, 
                label='History', color=COLORS.get('train', 'grey'), marker='.', linewidth=2)
        
        ax.plot(future_pred.index, future_pred.values, 
                linestyle=LINE_STYLES.get('pred', '--'), label='Forecast (VAR)', 
                color=COLORS.get('pred', 'red'), linewidth=2)
        
        # # Connessione History -> Prediction (per continuità visiva)
        # ax.plot([last_hist_idx, first_future_idx], [hist_series.iloc[-1], future_pred.iloc[0]], 
        #         linestyle=LINE_STYLES.get('pred', '--'), color=COLORS.get('pred', 'red'), linewidth=2)

        ax.plot(future_base.index, future_base.values, 
                linestyle=LINE_STYLES.get('baseline', ':'), label='Baseline (Drift)', 
                color=COLORS.get('baseline', 'blue'))
        
        # # Connessione History -> Baseline
        # ax.plot([last_hist_idx, first_future_idx], [hist_series.iloc[-1], future_base.iloc[0]], 
        #         linestyle=LINE_STYLES.get('baseline', ':'), color=COLORS.get('baseline', 'blue'))
        
        ax.axvspan(future_pred.index[0], future_pred.index[-1], color='#C3C3C3', alpha=0.3)
        
        long_name = REVERSE_VAR_NAMES.get(var_name, var_name)
        ax.set_title(f"Variable: {long_name}", fontsize=14, fontweight='bold')
        ax.set_ylabel("Value")
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend(loc='upper left')

    axes[1].set_xlabel("Year", fontsize=12)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=0)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    filename = f"VAR_{causing_var}_{caused_var}.png"
    full_path = os.path.join(save_folder, filename)
    try:
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved: {full_path}")
    except Exception as e:
        print(f"Error saving plot: {e}")
    
    plt.close()

In [60]:
TARGET_YEAR = 2030
BASELINE_METHOD = 'drift'
original_scaler_columns = list(df_causeeffect.columns)

last_date = df_causeeffect_scaled.index[-1]
years_to_predict = TARGET_YEAR - last_date.year
future_index = pd.date_range(
    start=last_date + pd.DateOffset(years=1),
    periods=years_to_predict,
    freq='YS'
)
print(f"Forecasting horizon: {years_to_predict} years (up to {TARGET_YEAR})")

future_results_level = {}
for pair_key, results in summary_results.items():
    causing, caused = pair_key
    print(f"Forecasting pair: {causing} -> {caused}")
    
    optimal_p = results['p_selected']
    try:
        model = VARMAX(df_causeeffect_scaled[[causing, caused]], order=(optimal_p, 0))
        res = model.fit(disp=False)
        forecast_scaled = res.forecast(steps=years_to_predict)
        
        if isinstance(forecast_scaled, pd.DataFrame):
            vals = forecast_scaled.values
        else:
            vals = forecast_scaled
            
    except Exception as e:
        print(f"Error fitting VAR for {pair_key}: {e}")
        continue

    df_future_scaled_diff = pd.DataFrame({
        f"{causing}_actual": [np.nan] * len(future_index),
        f"{caused}_actual": [np.nan] * len(future_index),
        f"{causing}_pred_VAR": vals[:, 0],
        f"{caused}_pred_VAR": vals[:, 1],
        f"{causing}_pred_baseline": [np.nan] * len(future_index),
        f"{caused}_pred_baseline": [np.nan] * len(future_index),
    }, index=future_index)
    
    df_future_unscaled = inverse_scale_dataframe(
        df_future_scaled_diff, scaler, original_scaler_columns, pair_key
    )
    
    df_future_level = inverse_difference_dataframe(
        df_future_unscaled, df, transformation_info, pair_key, len(df)
    )
    
    for var_name in [causing, caused]:
        train_serie = df[var_name].dropna()
        
        baseline_series = mod.get_baseline_prediction(
            BASELINE_METHOD,
            train_serie,
            future_index
        )
        col_base_name = f"{var_name}_pred_baseline"
        df_future_level[col_base_name] = baseline_series.values
        
    future_results_level[pair_key] = df_future_level
    
    plot_var_future_stacked(
        history_df=df,
        future_df=df_future_level,
        causing_var=causing,
        caused_var=caused,
        folder_name="06_FutureVAR"
    )

print("Future forecasting completed.")

Forecasting horizon: 9 years (up to 2030)
Forecasting pair: food_production_index -> valueadded_percent
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_FutureVAR\VAR_food_production_index_valueadded_percent.png
Forecasting pair: fertilizer_percent -> cerealland_abs
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_FutureVAR\VAR_fertilizer_percent_cerealland_abs.png
Forecasting pair: food_production_index -> fertilizer_percent
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_FutureVAR\VAR_food_production_index_fertilizer_percent.png
Forecasting pair: agriland_percent -> cerealland_abs
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_FutureVAR\VAR_agriland_percent_cerealland_abs.png
Forecasting pair: valueadded_percent -> fertilizer_percent
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\06_FutureV